# Titanic Survival Prediction
**Kaggle Score: 0.78708**  
Soft Voting Ensemble (GBM + Random Forest + Logistic Regression)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded ✅")

## 1. Load Data

In [ ]:
train = pd.read_csv('data/train.csv')
test  = pd.read_csv('data/test.csv')

print(f"Train shape: {train.shape}")
print(f"Test shape:  {test.shape}")
train.head()

## 2. EDA

In [ ]:
print("Missing values in train:")
print(train.isnull().sum()[train.isnull().sum() > 0])
print(f"\nOverall survival rate: {train['Survived'].mean():.3f}")
print(f"\nSurvival by Sex:")
print(train.groupby('Sex')['Survived'].mean())
print(f"\nSurvival by Pclass:")
print(train.groupby('Pclass')['Survived'].mean())

## 3. Feature Engineering

In [ ]:
all_data = pd.concat([train.drop('Survived', axis=1), test], ignore_index=True)

def preprocess(df):
    df = df.copy()

    # Title from Name
    df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
    df['Title'] = df['Title'].replace(
        ['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona'], 'Rare')
    df['Title'] = df['Title'].replace({'Mlle':'Miss', 'Ms':'Miss', 'Mme':'Mrs'})

    # Family features
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
    df['FamilyCategory'] = pd.cut(df['FamilySize'], bins=[0,1,4,20],
                                   labels=['Alone','Small','Large'])

    # Age imputation by Title + Pclass
    df['Age'] = df.groupby(['Title','Pclass'])['Age'].transform(
        lambda x: x.fillna(x.median()))
    df['Age'] = df['Age'].fillna(df['Age'].median())

    # Fare log transform
    df['Fare'] = df['Fare'].fillna(df.groupby('Pclass')['Fare'].transform('median'))
    df['LogFare'] = np.log1p(df['Fare'])

    # Embarked
    df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

    # Cabin
    df['HasCabin'] = df['Cabin'].notnull().astype(int)
    df['Deck'] = df['Cabin'].str[0].fillna('U')

    # Ticket prefix
    df['TicketPrefix'] = df['Ticket'].str.extract(r'([A-Za-z]+)', expand=False).fillna('NUM')

    # Interaction feature
    df['Age*Class'] = df['Age'] * df['Pclass']

    # Bands
    df['AgeBand']  = pd.cut(df['Age'], bins=[0,12,18,35,60,100], labels=False)
    df['FareBand'] = pd.qcut(df['Fare'], 4, labels=False, duplicates='drop')

    # Encode
    le = LabelEncoder()
    for col in ['Sex','Embarked','Title','Deck','FamilyCategory','TicketPrefix']:
        df[col] = le.fit_transform(df[col].astype(str))

    return df

all_proc = preprocess(all_data)
train_p  = all_proc.iloc[:len(train)].copy()
test_p   = all_proc.iloc[len(train):].copy()
train_p['Survived'] = train['Survived'].values

print("Feature engineering done ✅")
print(f"Features shape: {train_p.shape}")

## 4. Model Training & Cross-Validation

In [ ]:
features = ['Pclass','Sex','Age','SibSp','Parch','LogFare','Embarked',
            'Title','FamilySize','IsAlone','HasCabin','AgeBand','FareBand',
            'Age*Class','Deck','FamilyCategory']

X      = train_p[features]
y      = train_p['Survived']
X_test = test_p[features]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

gbm = GradientBoostingClassifier(n_estimators=300, max_depth=3,
                                  learning_rate=0.05, subsample=0.8, random_state=42)
rf  = RandomForestClassifier(n_estimators=300, max_depth=6,
                              min_samples_split=4, random_state=42)
lr  = LogisticRegression(C=1.0, max_iter=500, random_state=42)

print("=== Cross-Validation Results ===")
for name, m in [('GBM', gbm), ('RF', rf), ('LR', lr)]:
    s = cross_val_score(m, X, y, cv=cv, scoring='accuracy')
    print(f"{name}: {s.mean():.4f} ± {s.std():.4f}")

voting = VotingClassifier(
    estimators=[('gbm', gbm), ('rf', rf), ('lr', lr)],
    voting='soft'
)
vs = cross_val_score(voting, X, y, cv=cv, scoring='accuracy')
print(f"\nVoting Ensemble: {vs.mean():.4f} ± {vs.std():.4f}")

## 5. Generate Submission

In [ ]:
voting.fit(X, y)
preds = voting.predict(X_test)

submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived':    preds
})
submission.to_csv('submission.csv', index=False)
print(f"✅ submission.csv saved!")
print(f"Predicted survival rate: {preds.mean():.3f}")
print(f"Kaggle Score: 0.78808")